In [9]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

In [20]:
data = {
    "space": ["apartment","apartment","house","house","apartment","house","apartment","house","apartment","house",
              "apartment","house","apartment","house"],
    "time": ["low","medium","high","medium","low","high","medium","high","low","medium",
             "high","low","medium","high"],
    "budget": ["low","medium","high","medium","low","high","medium","high","low","medium",
               "high","low","medium","high"],
    "experience": ["beginner","beginner","experienced","experienced","beginner","experienced","beginner","experienced","beginner","experienced",
                   "experienced","beginner","experienced","beginner"],
    "pet": ["fish","cat","dog","dog","fish","dog","cat","dog","fish","cat",
            "dog","fish","cat","dog"]
}

df = pd.DataFrame(data)
df

,space,time,budget,experience,pet
0,apartment,low,low,beginner,fish
1,apartment,medium,medium,beginner,cat
2,house,high,high,experienced,dog
3,house,medium,medium,experienced,dog
4,apartment,low,low,beginner,fish
5,house,high,high,experienced,dog
6,apartment,medium,medium,beginner,cat
7,house,high,high,experienced,dog
8,apartment,low,low,beginner,fish
9,house,medium,medium,experienced,cat


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   space       14 non-null     object
 1   time        14 non-null     object
 2   budget      14 non-null     object
 3   experience  14 non-null     object
 4   pet         14 non-null     object
dtypes: object(5)
memory usage: 692.0+ bytes


In [22]:
df.isna().sum()

,0
space,0
time,0
budget,0
experience,0
pet,0


In [23]:
from sklearn.preprocessing import LabelEncoder

le_dict = {}

for col in df.columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le   # save encoders

df

,space,time,budget,experience,pet
0,0,1,1,0,2
1,0,2,2,0,0
2,1,0,0,1,1
3,1,2,2,1,1
4,0,1,1,0,2
5,1,0,0,1,1
6,0,2,2,0,0
7,1,0,0,1,1
8,0,1,1,0,2
9,1,2,2,1,0


In [24]:
from sklearn.model_selection import train_test_split

X = df.drop("pet", axis=1)
y = df["pet"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.fit(X_train, y_train)

DecisionTreeClassifier()

In [7]:
accuracy = model.score(X_test, y_test)
print("Model Accuracy:", accuracy)

Model Accuracy: 1.0


In [8]:
sample = [["apartment","low","low","beginner"]]

encoded_sample = []
for i, col in enumerate(["space","time","budget","experience"]):
    encoded_sample.append(le_dict[col].transform([sample[0][i]])[0])

prediction = model.predict([encoded_sample])


predicted_pet = le_dict["pet"].inverse_transform(prediction)

print("Recommended Pet:", predicted_pet[0])

Recommended Pet: fish


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


In [26]:
import gradio as gr

def predict_pet(space, time, budget, experience):
    # Encode the selected values
    encoded_space = le_dict['space'].transform([space])[0]
    encoded_time = le_dict['time'].transform([time])[0]
    encoded_budget = le_dict['budget'].transform([budget])[0]
    encoded_experience = le_dict['experience'].transform([experience])[0]

    sample_input = [[encoded_space, encoded_time, encoded_budget, encoded_experience]]

    # Make prediction
    prediction_encoded = model.predict(sample_input)
    predicted_pet = le_dict['pet'].inverse_transform(prediction_encoded)

    return predicted_pet[0]

# Get options for dropdowns from the LabelEncoders
space_options = list(le_dict['space'].inverse_transform(sorted(df['space'].unique())))
time_options = list(le_dict['time'].inverse_transform(sorted(df['time'].unique())))
budget_options = list(le_dict['budget'].inverse_transform(sorted(df['budget'].unique())))
experience_options = list(le_dict['experience'].inverse_transform(sorted(df['experience'].unique())))

# Create Gradio interface
iface = gr.Interface(
    fn=predict_pet,
    inputs=[
        gr.Dropdown(space_options, label="Space"),
        gr.Dropdown(time_options, label="Time"),
        gr.Dropdown(budget_options, label="Budget"),
        gr.Dropdown(experience_options, label="Experience")
    ],
    outputs="text",
    title="Pet Recommendation System",
    description="Select your preferences to get a pet recommendation."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d4bd1d21d577c471d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
